In [1]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, StructField, IntegerType, BooleanType, DateType
from datetime import date

minio_endpoint = "http://172.16.0.4:9000"
spark = (
    SparkSession.builder
    .appName("MinIOExample")
    .config(
        "spark.jars.packages",
        ",".join(["org.apache.hadoop:hadoop-aws:3.3.2", 
                  "com.amazonaws:aws-java-sdk-bundle:1.12.262",
                  "io.delta:delta-spark_2.12:3.2.0"])
    )
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

In [2]:
# Lendo as informações que foram armazenadas no bucket
transactions_raw = "s3a://bronze/datalake/raw/transactions/"
df_dados = spark.read.format('csv').options(header=True, inferSchema=True).load(transactions_raw)

In [ ]:
df_dados.select("amount").describe().show()

In [3]:
# DATA PROFILING
print("Analisando a distribuição da coluna amount")
df_dados.select("amount").describe().show()
print("\nAnalisando quantidade de valores nulos por coluna:")
print(f'transaction_id: {df_dados.where("transaction_id is null").count()}')
print(f'tx_date: {df_dados.where("tx_date is null").count()}')
print(f'state: {df_dados.where("state is null").count()}')
print(f'amount: {df_dados.where("amount is null").count()}')
print(f'status: {df_dados.where("status is null").count()}')

# DEFININDO O CONTRATO DE DADOS
# transaction_id NÃO PODE SER NULO
# amount > 0
# status IN ('ACTIVE', 'PENDENT', 'CANCELLED')
# state IN ('RJ', 'MG', 'SP', 'ES', 'PR')

rule_valid_transaction_id = F.col("transaction_id").isNotNull()
rule_valid_amount = F.col("amount") > 0
rule_valid_status = F.col("status").isin(['ACTIVE', 'PENDENT', 'CANCELLED'])
rule_valid_state = F.col("state").isin(['RJ', 'MG', 'SP', 'ES', 'PR'])
print("\nAplicando as regras do contrato de dados:")
df_dados = df_dados.withColumn("analise_contrato_dados", rule_valid_transaction_id & rule_valid_amount & rule_valid_status & rule_valid_state)
df_dados.show()
df_quarentena_dados = df_dados.where("analise_contrato_dados == false")
df_dados = df_dados.where("analise_contrato_dados == true")

Analisando a distribuição da coluna amount
+-------+------------------+
|summary|            amount|
+-------+------------------+
|  count|            102065|
|   mean|2448.0774027831203|
| stddev|1443.9831338340766|
|    min|-49.99074590639469|
|    max| 4949.999664521441|
+-------+------------------+


Analisando quantidade de valores nulos por coluna:
transaction_id: 5060
tx_date: 0
state: 0
amount: 0
status: 0

Aplicando as regras do contrato de dados:
+--------------+----------+-----+------------------+---------+----------------------+
|transaction_id|   tx_date|state|            amount|   status|analise_contrato_dados|
+--------------+----------+-----+------------------+---------+----------------------+
|         50001|2026-05-25|   SP| 413.3623826534183|  CORRUPT|                 false|
|         50002|2026-09-12|   XX|2652.0589183024363|CANCELLED|                 false|
|         50003|2026-04-26|   XX| 4079.655995348301|   ACTIVE|                 false|
|         50004|2026-02

In [ ]:
df_dados.show()

In [4]:
# DEFININDO LOCAIS DE SALVAMENTO NA CAMADA SILVER
transactions_silver = "s3a://silver/datalake/transactions/dados"
quarentena_transactions_silver = "s3a://silver/datalake/quarentena/transactions/dados"

# Salvando os dados brutos no MinIO (Camada Silver)
print(f"Gravando os registros válidos no Datalake {transactions_silver}")
df_dados.write.mode("overwrite").format("delta").save(transactions_silver)

print(f"Gravando os registros inválidos no Datalake (quarentena) {quarentena_transactions_silver}")
df_quarentena_dados.write.mode("overwrite").format("delta").save(quarentena_transactions_silver)

Gravando os registros válidos no Datalake s3a://silver/datalake/transactions/dados
Gravando os registros inválidos no Datalake (quarentena) s3a://silver/datalake/quarentena/transactions/dados


In [5]:
new_schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("tx_date", DateType(), True),
    StructField("state", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("analise_contrato_dados", BooleanType(), True),
    StructField("device_id", StringType(), True)
])
data_new_schema = [(100000, date(2026,10,26), "MG", 1587.98, "ACTIVE", True, "Android Phone")]
df_new_data = spark.createDataFrame(data_new_schema, new_schema)
df_new_data.write.mode("append").format("delta").option("mergeSchema", "true").save(transactions_silver)

In [ ]:
df_new_data.write.mode("append").format("delta").option("mergeSchema", "true").save(transactions_silver)

In [ ]:
df_dados = spark.read.format('delta').load(transactions_silver)

In [ ]:
df_dados.show()

In [7]:
delta_transactions = spark.read.format('delta').load(transactions_silver)
delta_transactions.createOrReplaceTempView("transactions")
# Para visualizar os dados
spark.sql("select * from transactions").show()
# Para atualizar os dados
spark.sql("update transactions set device_id = 'iOS phone' WHERE transaction_id = 115")
# Exibir os dados novamente
spark.sql("select * from transactions").show()

+--------------+----------+-----+------------------+---------+----------------------+---------+
|transaction_id|   tx_date|state|            amount|   status|analise_contrato_dados|device_id|
+--------------+----------+-----+------------------+---------+----------------------+---------+
|           115|2026-01-26|   PR| 4806.856266145014|   ACTIVE|                  true|     NULL|
|           126|2026-07-16|   SP|3947.7314078561594|CANCELLED|                  true|     NULL|
|           422|2026-07-26|   SP| 3565.557039520701|   ACTIVE|                  true|     NULL|
|           448|2026-07-01|   RJ|3916.7998001611495|   ACTIVE|                  true|     NULL|
|           528|2026-02-27|   ES|2789.1138669328116|   ACTIVE|                  true|     NULL|
|           548|2026-06-02|   ES|   2524.1706487267|   ACTIVE|                  true|     NULL|
|           555|2026-01-15|   ES| 4274.400329522427|CANCELLED|                  true|     NULL|
|           555|2026-01-15|   ES| 4274.4